# Import Qwen

In [1]:
%load_ext autoreload

%autoreload 2

import warnings

# Ignore all FutureWarnings to clean up your notebook outputs
warnings.filterwarnings("ignore", category=FutureWarning)

# Annoying problems with unsloth, it requires import unsloth before torch
import unsloth
import torch

from app.graph.workflow import build_graph
from langchain_core.messages import HumanMessage, ToolMessage
from IPython.display import Image, display

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/nguyen/micromamba/envs/unsloth_fresh/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

# Test Graph Workflow

In [ ]:
from app.graph.workflow import build_graph
from app.core.app import container
from pprint import pprint as ppr
from app.memory.extractor import MemoryExtractor

graph = build_graph()

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="What is FAISS? I want detail abou it!"
            )
        ]
    }
)

display(Image(graph.get_graph().draw_mermaid_png()))

ppr(result)

OperationalError: no such table: memories

In [ ]:
extracted_memories = container.memory_extractor.extract(result["messages"])
ppr(extracted_memories)

Both `max_new_tokens` (=2048) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
container.memory_manager.store.save(extracted_memories)

faiss_items = []

for memory in extracted_memories:

    faiss_id = (
        container.memory_store
        .get_next_faiss_id()
    )

    container.memory_store.save_faiss_mapping(
        faiss_id=faiss_id,
        memory_id=memory.id,
    )

    faiss_items.append(
        (
            faiss_id,
            memory.content,
        )
    )

container.faiss_store.add_many(
    faiss_items
)

container.faiss_store.save()

Traceback (most recent call last):
  File "_pydevd_bundle\\pydevd_cython.pyx", line 1609, in _pydevd_bundle.pydevd_cython.handle_exception
  File "/home/nguyen/.local/lib/python3.12/site-packages/debugpy/_vendored/pydevd/pydevd.py", line 2199, in do_wait_suspend
    keep_suspended = self._do_wait_suspend(thread, frame, event, arg, trace_suspend_type, from_this_thread, frames_tracker)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nguyen/.local/lib/python3.12/site-packages/debugpy/_vendored/pydevd/pydevd.py", line 2268, in _do_wait_suspend
    notify_event.wait(wait_timeout)
  File "/home/nguyen/micromamba/envs/unsloth_fresh/lib/python3.12/threading.py", line 655, in wait
    signaled = self._cond.wait(timeout)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nguyen/micromamba/envs/unsloth_fresh/lib/python3.12/threading.py", line 359, in wait
    gotit = waiter.acquire(True, timeout)
        

NameError: name 'container' is not defined

: 

In [23]:
import faiss
import random
import uuid

import numpy as np

# 1. Initialize a base index wrapped inside an ID Map
dimension = 1536  # e.g., OpenAI embeddings
base_index = faiss.IndexFlatIP(dimension)
index = faiss.IndexIDMap(base_index)

# 2. When a new memory comes in:
memory_text = "User prefers light-weight environments like Micromamba."
memory_uuid = str(uuid.uuid4())  # Generates sortable string: '018fbc...'

# Create a deterministic int64 for FAISS using a fast hash function
# (or pull an explicit bigint surrogate key from your bridge table)
faiss_id = hash(memory_uuid) & 0x7FFFFFFFFFFFFFFF  # Ensure safe positive int64

print(faiss_id)

# 3. Save to storage
# SQLite stores: memory_uuid (TEXT), memory_text (TEXT), faiss_id (INTEGER)
# FAISS stores: vector mapped to faiss_id
vector = np.random.rand(1, dimension).astype('float32')
print(vector)
index.add_with_ids(vector, np.array([faiss_id], dtype=np.int64))
print(index)

1553128298816415984
[[0.28249314 0.858315   0.36576033 ... 0.9832585  0.00745251 0.49562252]]
<faiss.swigfaiss.IndexIDMap; proxy of <Swig Object of type 'faiss::IndexIDMapTemplate< faiss::Index > *' at 0x7163025c1770> >


# Any Tests

In [7]:
from datetime import UTC, datetime

print(datetime.now(UTC))

2026-06-23 15:20:50.664584+00:00


# Test Schemas

In [ ]:
from app.tools.registry import ToolRegistry
from pprint import pprint as ppr

registry = ToolRegistry(register_all_available=True)

ppr(registry.get_tool_schemas())


# Test Model generation

In [9]:
from app.core.app import container
from pprint import pprint as ppr
from langchain_core.messages import HumanMessage

ppr(container.model.invoke(
    [HumanMessage(content="What is the capital of Vietnam?")]
))

Both `max_new_tokens` (=1024) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AgentDecision(thought='The capital of Vietnam is a straightforward factual question.', response='Hanoi', tool_calls=[])
